In [1]:
import random
import matplotlib.pyplot as plt
from src.utils import *
from src.preprocessing_teresa import *
from src.preprocessing import *
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# Group all paths by serie and pick one randomly to test
paths = get_all_image_paths()
series = group_paths_by_serie(paths)
print(f"Total series found: {len(series)}")

random_serie_key = random.choice(list(series.keys()))
random_serie_paths = series[random_serie_key]
print(f"Selected serie : {random_serie_key}")
print(f"Frames in serie: {len(random_serie_paths)}")

SEED = 33

Total images found: 8325
Total series found: 214
Selected serie : 031/6
Frames in serie: 46


---
## Step 1 — Background Subtraction
Before CLAHE, we remove the slow-varying background illumination (ribcage, diaphragm, soft tissue gradients) using a large Gaussian blur:

```
background = GaussianBlur(image, large_kernel)
residual   = image − background   → vessels on a flat background
output     = rescale to [0, 255]  → ready for CLAHE
```

Use the **kernel size** slider to control how aggressively the background is estimated.
- Too small → vessels start getting absorbed into the background estimate
- Too large → background not fully removed, uneven illumination remains
- Sweet spot → flat grey background with vessels clearly standing out

In [ ]:
browse_multiscale_residual_grid(
    series,
    n=9,
    seed=SEED
)

interactive(children=(FloatSlider(value=1.0, description='Residual amplification α', layout=Layout(width='450p…

browse_background_vs_gamma(series, n=9, seed=SEED)

In [ ]:
browse_background_vs_gamma(series, n=9, seed=SEED)

interactive(children=(IntSlider(value=21, description='BG kernel size', layout=Layout(width='500px'), max=201,…

GLOBAL_REFERENCE = compute_global_reference(
    paths,
    n_samples=50,
    seed=42
)

browse_background_vs_histmatch(
    series,
    global_reference=GLOBAL_REFERENCE,
    n=9,
    seed=SEED
)

In [ ]:
GLOBAL_REFERENCE = compute_global_reference(
    paths,
    n_samples=50,
    seed=42
)

browse_background_vs_histmatch(
    series,
    global_reference=GLOBAL_REFERENCE,
    n=9,
    seed=SEED
)

Global reference computed from 50 images.


interactive(children=(IntSlider(value=21, description='BG kernel size', layout=Layout(width='500px'), max=201,…

browse_residual_fusion_grid(
    series,
    n=9,
    seed=SEED
)

In [ ]:
browse_residual_fusion_grid(
    series,
    n=9,
    seed=SEED
)

interactive(children=(IntSlider(value=31, description='Kernel size', layout=Layout(width='450px'), max=201, mi…

##  Step 2 — CLAHE
Apply CLAHE using different clip limits to see the effect on contrast enhancement. The clip limit controls how much the histogram is allowed to be amplified, which can help prevent over-enhancement of noise in homogeneous areas.


A 3×3 grid of **one middle frame per patient** (randomly selected).
Use the single slider to adjust the clip limit and assess which value generalises best across different patients — left column is always the original, right column the CLAHE result.

In [ ]:
KERNEL_SIZE = 21  # adjust manually and re-run

browse_bg_clahe_grid(series, kernel_size=KERNEL_SIZE, n=9, seed=SEED)

interactive(children=(FloatSlider(value=2.0, description='Clip limit', layout=Layout(width='450px'), max=10.0,…

## Step 3 — NLM Filtering
BG subtraction and CLAHE are fixed from the steps above — set `KERNEL_SIZE` and `CLIP_LIMIT` to match.

NLM has three parameters to tune:
- **h** — filter strength: the main knob. Higher = more smoothing, more vessel edge loss. Try 5–15.
- **Template window** — patch size for comparing pixel similarity (must be odd). Default 7 is usually fine.
- **Search window** — area searched for similar patches (must be odd). Larger = better but slower. Default 21 is usually fine.

Start by tuning **h** with template/search windows at defaults, then adjust the others only if needed.

KERNEL_SIZE = 21  # adjust manually and re-run
CLIP_LIMIT  = 2.0  # must match the value used in Step 3
browse_nlm_grid(series, kernel_size=KERNEL_SIZE, clip_limit=CLIP_LIMIT, n=9, seed=SEED)

## Step 4b — Bilateral Filtering (alternative to NLM)
Same fixed BG subtraction + CLAHE as above. Three parameters to tune:

- **d** — neighborhood diameter. Higher = larger area considered per pixel.
- **sigma_color** — intensity tolerance. Low = only very similar intensities averaged (edge-preserving). High = smoother but edges start blurring.
- **sigma_space** — spatial tolerance. Higher = farther pixels contribute more.

A good starting point is d=9, sigma_color=75, sigma_space=75. Raise sigma_color to increase smoothing; lower it to preserve more vessel edge detail.

# Use the same NLM parameters to compare results.
KERNEL_SIZE = KERNEL_SIZE
CLIP_LIMIT  = CLIP_LIMIT
browse_bilateral_grid(series, kernel_size=KERNEL_SIZE, clip_limit=CLIP_LIMIT, n=9, seed=42)